
# Error Type Detection using BiLSTM

Notebook ini:
1. Menggabungkan seluruh dataset IPA, IPS, dan Bahasa Indonesia.
2. Melatih model BiLSTM untuk mendeteksi jenis kesalahan.
3. Memprediksi salah satu dari 6 kelas:
   - Deletion Error
   - Insertion Error
   - Punctuation Error
   - Real Word Error
   - Substitution Error
   - Transposition Error
4. Menampilkan highlight kesalahan pada teks.


In [1]:
!pip install pandas numpy torch transformers language-tool-python datasets lxml
!pip install datasets kaggle
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d yanfiyanfi/specil-spell-error-corpus-for-indonesian-language
!unzip -o specil-spell-error-corpus-for-indonesian-language.zip
!ls
!pip install transformers datasets sentencepiece accelerate evaluate
!pip install evaluate jiwer rouge-score sacrebleu
!wget https://dumps.wikimedia.org/idwiki/latest/idwiki-latest-pages-articles.xml.bz2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 4.1 MB/s eta 0:00:00
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/yanfiyanfi/specil-spell-error-corpus-for-indonesian-language
License(s): other
100% 11.7M/11.7M [00:00<00:00, 102MB/s] 

Archive:  specil-spell-error-corpus-for-indonesian-language.zip
  inflating: BIndo_output_deletion_error.csv  
  inflating: BIndo_output_insertion_error.csv  
  inflating: BIndo_output_punctuation_error.csv  
  inflating: BIndo_output_real-word_error.csv  
  inflating: BIndo_output_subtitution_error.csv  
  inflating: BIndo_output_transposition_error.csv  
  inflating: IPA_output_deletion_error.csv  
  inflating: IPA_output_insertion_error.csv  
  inflating: IPA_output_punctuation_error.csv  
  inflating: IPA_output_real-word_error.csv  
  inflating: IPA_output_subtitution_error.csv  
  inflating: IPA_output_transpo

In [2]:
import pandas as pd
import numpy as np
import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical


## Load dan Gabungkan Semua CSV

In [3]:

files = [
    "BIndo_output_deletion_error.csv",
    "BIndo_output_insertion_error.csv",
    "BIndo_output_punctuation_error.csv",
    "BIndo_output_real-word_error.csv",
    "BIndo_output_subtitution_error.csv",
    "BIndo_output_transposition_error.csv",
    "IPA_output_deletion_error.csv",
    "IPA_output_insertion_error.csv",
    "IPA_output_punctuation_error.csv",
    "IPA_output_real-word_error.csv",
    "IPA_output_subtitution_error.csv",
    "IPA_output_transposition_error.csv",
    "IPS_output_deletion_error.csv",
    "IPS_output_insertion_error.csv",
    "IPS_output_punctuation_error.csv",
    "IPS_output_real-word_error.csv",
    "IPS_output_subtitution_error.csv",
    "IPS_output_transposition_error.csv"
]

dfs = []

for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print(data.shape)
data.head()


(387000, 4)


,kalimat_awal,kalimat_salah,Pelajaran,tipe_kesalahan
0,Bunyi apa?,Buni apa?,Bahasa Indonesia,Deletion Error
1,Siap-siap belajar.,Sia-siap belajar.,Bahasa Indonesia,Deletion Error
2,Diskusikan gambar sampul di atas dengan menjaw...,Diskusikan gambar sampul di atas dengan menawa...,Bahasa Indonesia,Deletion Error
3,Apa yang kalian lihat pada gambar di atas?,Apa yang kalian lihat pada gambar d atas?,Bahasa Indonesia,Deletion Error
4,"Menurut kalian, apa isi ceritanya?","Menurut kalian, apa si ceritanya?",Bahasa Indonesia,Deletion Error


In [4]:

data = data.dropna()

correct_data = pd.DataFrame({
    "kalimat_salah": data["kalimat_awal"],
    "kalimat_awal": data["kalimat_awal"],
    "tipe_kesalahan": "correct"
})

data = pd.concat(
    [data, correct_data],
    ignore_index=True
)

data = data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(data["tipe_kesalahan"].value_counts())

data["kalimat_salah"] = data["kalimat_salah"].astype(str)
data["tipe_kesalahan"] = data["tipe_kesalahan"].astype(str)

data = data.sample(frac=1, random_state=42).reset_index(drop=True)

data.head()


tipe_kesalahan
correct                387000
Subtitution Error       64500
Punctuation Error       64500
Insertion Error         64500
Real word Error         64500
Transposition Error     64500
Deletion Error          64500
Name: count, dtype: int64


,kalimat_awal,kalimat_salah,Pelajaran,tipe_kesalahan
0,Kenampakan alam myanmar berupa pegunungan dan ...,Kenampakan alam myanmar berupa pegunungan dan ...,IPS,Punctuation Error
1,Penghargaan bergengsi yang pernah diraih habib...,Penghargaan bergengsi yang pernah diraih habib...,NaN,correct
2,Selain itu di dataran tinggi sering berkabut.,Selain itu di dataran tinggal sering berkabut.,IPS,Real word Error
3,Flu burung disebabkan oleh virus pada unggas.,Flu burung disebabkan oleh virus pada unggtas.,IPA,Insertion Error
4,Selama hampir satu minggu kudus bagaikan kota ...,Selama hampir satu minggu kudus bagaikan kota ...,Bahasa Indonesia,Punctuation Error


## Label Encoding

In [5]:
encoder = LabelEncoder()

data["label"] = encoder.fit_transform(
    data["tipe_kesalahan"]
)

print(encoder.classes_)

['Deletion Error' 'Insertion Error' 'Punctuation Error' 'Real word Error'
 'Subtitution Error' 'Transposition Error' 'correct']


## Tokenisasi

In [6]:

MAX_LEN = 50
VOCAB_SIZE = 30000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<UNK>"
)

tokenizer.fit_on_texts(
    data["kalimat_salah"]
)

X = tokenizer.texts_to_sequences(
    data["kalimat_salah"]
)

X = pad_sequences(
    X,
    maxlen=MAX_LEN,
    padding="post"
)

y = to_categorical(
    data["label"]
)

print(X.shape)
print(y.shape)


(774000, 50)
(774000, 7)


In [7]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)


## Model BiLSTM

In [8]:

model = Sequential([

    Embedding(
        VOCAB_SIZE,
        128,
        input_length=MAX_LEN
    ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=False
        )
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        len(encoder.classes_),
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=3,
    batch_size=64
)


Epoch 1/3
9675/9675 ━━━━━━━━━━━━━━━━━━━━ 2784s 287ms/step - accuracy: 0.7892 - loss: 0.6100 - val_accuracy: 0.8446 - val_loss: 0.4391
Epoch 2/3
9675/9675 ━━━━━━━━━━━━━━━━━━━━ 2815s 291ms/step - accuracy: 0.8530 - loss: 0.4034 - val_accuracy: 0.8582 - val_loss: 0.3942
Epoch 3/3
9675/9675 ━━━━━━━━━━━━━━━━━━━━ 2755s 285ms/step - accuracy: 0.8622 - loss: 0.3679 - val_accuracy: 0.8592 - val_loss: 0.3831


## Evaluasi

In [10]:
loss, acc = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:", acc)

4838/4838 ━━━━━━━━━━━━━━━━━━━━ 234s 48ms/step - accuracy: 0.8592 - loss: 0.3831
Accuracy: 0.859205424785614


## Highlight Error

In [11]:

from difflib import SequenceMatcher

def highlight_errors(correct_text, wrong_text):

    correct_words = correct_text.split()
    wrong_words = wrong_text.split()

    result = []

    matcher = SequenceMatcher(
        None,
        correct_words,
        wrong_words
    )

    for tag,i1,i2,j1,j2 in matcher.get_opcodes():

        if tag == "equal":
            result.extend(
                wrong_words[j1:j2]
            )

        else:
            for word in wrong_words[j1:j2]:
                result.append(
                    f"[ERROR:{word}]"
                )

    return " ".join(result)


## Prediksi Jenis Kesalahan

In [12]:

from difflib import SequenceMatcher

ERROR_EXPLANATION = {

    "correct":
    "Kalimat tidak terdeteksi memiliki kesalahan.",

    "deletion_error":
    "Ada karakter atau kata yang hilang.",

    "insertion_error":
    "Ada karakter atau kata tambahan yang tidak seharusnya ada.",

    "punctuation_error":
    "Terdapat kesalahan tanda baca.",

    "real-word_error":
    "Kata valid tetapi digunakan secara tidak tepat.",

    "subtitution_error":
    "Ada karakter atau kata yang tertukar dengan yang lain.",

    "transposition_error":
    "Urutan karakter atau kata tertukar."
}

def extract_error_detail(correct_text, wrong_text):

    c_words = correct_text.split()
    w_words = wrong_text.split()

    matcher = SequenceMatcher(None, c_words, w_words)

    wrong_part = []
    correct_part = []

    for tag,i1,i2,j1,j2 in matcher.get_opcodes():
        if tag != "equal":
            wrong_part.extend(w_words[j1:j2])
            correct_part.extend(c_words[i1:i2])

    return " ".join(wrong_part), " ".join(correct_part)

def predict_error(text, correct_text=None):

    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=MAX_LEN, padding="post")

    pred = model.predict(seq, verbose=0)

    idx = np.argmax(pred)
    label = encoder.inverse_transform([idx])[0]

    result = {
      "status":
          "BENAR"
          if label == "correct"
          else "SALAH",

      "error_type": label,

      "confidence": float(np.max(pred)),

      "explanation":
          ERROR_EXPLANATION.get(
            label,
            "Jenis kesalahan tidak diketahui."
          )
    }

    if correct_text is not None:
        wrong_text, correct_part = extract_error_detail(
            correct_text,
            text
        )

        result["wrong_text"] = wrong_text
        result["correct_text"] = correct_part

    return result


In [13]:

row = data.sample(1).iloc[0]

result = predict_error(
    row["kalimat_salah"],
    row["kalimat_awal"]
)

print(result)


{'status': 'SALAH', 'error_type': 'Insertion Error', 'confidence': 0.9999637603759766, 'explanation': 'Jenis kesalahan tidak diketahui.', 'wrong_text': 'lebith', 'correct_text': 'lebih'}


## Prediksi + Highlight dari Dataset

In [14]:

row = data.sample(1).iloc[0]

print("Kalimat Awal :")
print(row["kalimat_awal"])

print("\nKalimat Salah :")
print(row["kalimat_salah"])

print("\nHighlight :")
print(
    highlight_errors(
        row["kalimat_awal"],
        row["kalimat_salah"]
    )
)

print("\nPrediksi Model :")
print(
    predict_error(
    row["kalimat_salah"],
    row["kalimat_awal"]
  )
)


Kalimat Awal :
Merefleksikan hasil belajar bab 2 untuk mengetahui topik yang sudah berhasil dipahami dengan baik dan yang perlu dikuasai lebih lanjut.

Kalimat Salah :
Merefleksikan hasil belajar bab 2 untuk mengetahui topik yang sudah berhasil dipahami dengan baik dan yang perlu dikuasai letih lanjut.

Highlight :
Merefleksikan hasil belajar bab 2 untuk mengetahui topik yang sudah berhasil dipahami dengan baik dan yang perlu dikuasai [ERROR:letih] lanjut.

Prediksi Model :
{'status': 'SALAH', 'error_type': 'Real word Error', 'confidence': 0.9964672327041626, 'explanation': 'Jenis kesalahan tidak diketahui.', 'wrong_text': 'letih', 'correct_text': 'lebih'}


In [15]:
model.save("bilstm_error_model.h5")

In [16]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer berhasil disimpan")

Tokenizer berhasil disimpan


In [17]:
import pickle

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Label encoder berhasil disimpan")

Label encoder berhasil disimpan


In [18]:
model.save("bilstm_error_model.keras")

In [19]:
import tensorflow as tf

print(tf.__version__)

2.20.0
